# Pipeline V8 — Fase 3: Rotulação/Re-rotulação Local (CPU Multicore)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-007). **Execução: PC local.**

Versão local da `fase3_rerotulacao_databricks.ipynb` para rodar em CPU com
multiprocessamento Python (`multiprocessing.Pool.imap_unordered`).
Otimizada para Ryzen 5700X (8 cores / 16 threads); funciona em qualquer multicore.

Calcula `melhor_jogada`, `score_melhor_jogada` e `depth_melhor_jogada` para
qualquer lote de NPZs com schema v2-a3 (12 canais + 3 escalares de cadeias).

**Diferenças em relação à versão Databricks:**
- Sem PySpark: usa `multiprocessing.Pool` com worker em `_worker_fase3_local.py`.
- `INPUT_DIR` aponta para o diretório local de dados (configurável na célula abaixo).
- `NUM_WORKERS` = `cpu_count() − 2` por padrão (ajuste conforme necessário).
- `LOTE_ARQUIVOS` = 8 (checkpoints mais frequentes que o Databricks).

**Cache global cross-NPZ (melhoria sobre o Databricks):**
Antes de processar, o notebook varre todos os NPZs já rotulados e constrói um
`cache_global` com os resultados já calculados. Tabuleiros idênticos em NPZs novos
recebem `melhor_jogada`, `score_melhor_jogada` e `depth_melhor_jogada` diretamente
do cache — sem recomputar o Minimax. Novos resultados computados também são
adicionados ao cache e aproveitados pelos lotes seguintes.

### Estratégia de profundidade adaptativa

```
depth_estado = DEPTH_ADAPTATIVO  se  arestas_livres > 11  E  prof_min > 11
             = DEPTH_PADRAO       caso contrário

onde:  arestas_livres = 31 − qtd_tracos
       prof_min       = total_caixas_cadeias_longas + 2 × qtd_cadeias_longas
```

O Minimax para naturalmente; `DEPTH_ADAPTATIVO=20` resolve qualquer estado com
≤ 19 arestas livres (máximo observado no dataset).

### Requisito: `_worker_fase3_local.py`

O worker deve estar no **mesmo diretório** que este notebook. Toda a lógica
Minimax fica lá — ambos os bug fixes de Maio/2026 estão aplicados.


> **⚠️ ALERTA DE MANUTENÇÃO: BUGS ALGÓRITMICOS NO BITBOARD (Maio/2026)**
>
> O Minimax via Bitboard no `_worker_fase3_local.py` possui dois bugs que foram
> corrigidos e **NÃO DEVEM ser revertidos ou esquecidos** ao adaptar este código:
>
> **Bug 1 — Caixas Pré-Fechadas (Falsos Positivos):** ao contar caixas completadas
> por uma jogada, é OBRIGATÓRIO verificar que a caixa *não estava fechada antes*:
> ```python
> # CORRETO:
> closed = sum(1 for bm in EDGE_BOXES[i] if new_e & bm == bm and edges & bm != bm)
> # ERRADO (conta caixas já fechadas antes da jogada):
> closed = sum(1 for bm in EDGE_BOXES[i] if new_e & bm == bm)
> ```
>
> **Bug 2 — Offsets na Poda Alpha-Beta Incremental:** este Bitboard retorna scores
> *incrementais* (não absolutos). Ao chamar a subárvore após capturar `closed` caixas,
> os limites alpha/beta DEVEM ser ajustados com offset de `−closed`:
> ```python
> # CORRETO (maximizando, capturou `closed` caixas — mesma vez):
> val = closed + solve_minimax_bitboard(new_e, depth-1, alpha - closed, beta - closed, True, tt)
> # ERRADO (repassa alpha/beta puro — causa poda prematura e escolhe jogadas perdedoras):
> val = closed + solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
> ```


In [1]:
import os
import sys
import time
import multiprocessing
from pathlib import Path

import numpy as np

try:
    from tqdm.auto import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    print('[INFO] tqdm nao instalado — sem barra de progresso. pip install tqdm')

# ── Localiza raiz do repositório ──────────────────────────────────────────────
ROOT = Path().resolve()
while not (ROOT / 'gerador_dados').exists():
    ROOT = ROOT.parent
    if ROOT == ROOT.parent:
        raise RuntimeError('Raiz do repositorio nao encontrada (pasta gerador_dados ausente).')

V8_DIR = ROOT / 'gerador_dados' / 'jogo_pontinhos' / 'v8'
if str(V8_DIR) not in sys.path:
    sys.path.insert(0, str(V8_DIR))

from _worker_fase3_local import processar_estado  # noqa: E402

# ── Configurações ─────────────────────────────────────────────────────────────
# Ajuste INPUT_DIR para o diretório dos NPZs que deseja rotular.
INPUT_DIR = ROOT / 'dados' / 'profundidade_minimax_11_adaptativo'

DEPTH_PADRAO     = 11
DEPTH_ADAPTATIVO = 20

# NPZs por lote (checkpoint após cada lote gravado).
LOTE_ARQUIVOS = 1

# Ryzen 5700X: 8 cores / 16 threads. Deixa 2 para o SO + processo principal.
NUM_WORKERS = max(1, multiprocessing.cpu_count() - 2)

# Se True: re-rotula TODOS os estados, inclusive os já rotulados.
FORCAR_REGRAVAR = False

print(f'ROOT             = {ROOT}')
print(f'INPUT_DIR        = {INPUT_DIR}')
print(f'DEPTH_PADRAO     = {DEPTH_PADRAO}')
print(f'DEPTH_ADAPTATIVO = {DEPTH_ADAPTATIVO}')
print(f'LOTE_ARQUIVOS    = {LOTE_ARQUIVOS}')
print(f'NUM_WORKERS      = {NUM_WORKERS}  (cpu_count={multiprocessing.cpu_count()})')
print(f'FORCAR_REGRAVAR  = {FORCAR_REGRAVAR}')
print(f'Worker           : _worker_fase3_local.py')


ROOT             = D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend
INPUT_DIR        = D:\Desenvolvimento\arena-sagaz\arena-sagaz-backend\dados\profundidade_minimax_11_adaptativo
DEPTH_PADRAO     = 11
DEPTH_ADAPTATIVO = 20
LOTE_ARQUIVOS    = 1
NUM_WORKERS      = 14  (cpu_count=16)
FORCAR_REGRAVAR  = False
Worker           : _worker_fase3_local.py


In [2]:
# ── Diagnóstico + pré-carga do cache global ───────────────────────────────────
# Varredura ÚNICA em todos os NPZs:
#   1. Conta estados totais / pendentes (bruto e distintos).
#   2. Constrói cache_global com resultados já calculados em NPZs rotulados.
#      Tabuleiros idênticos em NPZs novos receberão os dados do cache sem
#      recomputar o Minimax.
#
# Memória: cache_global ocupa ~200-250 MB para 760 k estados rotulados.
# pending_hashes usa hash de 64 bits — colisão improvável (~0,06% p/ 2 M itens).

SUFIXOS_SIMETRIA = ('_refH', '_refV', '_r180')
todos_npzs = sorted(INPUT_DIR.glob('dataset_pequeno_*.npz'))
npzs_originais = [
    p for p in todos_npzs
    if not any(p.stem.endswith(s) for s in SUFIXOS_SIMETRIA)
    and '.tmp.' not in p.name
]

# cache_global: (estado_bytes, depth_necessaria) -> (melhor_jogada, scores_array)
cache_global: dict = {}

# Lista de NPZs com ao menos um estado pendente.
pendentes: list = []

# Hashes dos pares (estado_bytes, depth_necessaria) pendentes.
# Usa int de 64 bits em vez de bytes para economizar ~85% de memória.
pending_hashes: set = set()

n_padrao = 0
n_adaptativo = 0
n_pendentes_padrao = 0
n_pendentes_adaptativo = 0

print(f'Escaneando {len(npzs_originais)} NPZs (diagnóstico + cache)...')
t0 = time.time()

for path in npzs_originais:
    with np.load(path, allow_pickle=False) as d:
        estados_arr = d['estados']
        qtd_tracos  = d['qtd_tracos'].astype(np.int32)
        qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_atual = (
            d['depth_melhor_jogada'].astype(np.int32)
            if 'depth_melhor_jogada' in d.files
            else np.zeros(len(qtd_tracos), dtype=np.int32)
        )
        rotulados = (
            (d['melhor_jogada'] != '')
            if 'melhor_jogada' in d.files
            else np.zeros(len(qtd_tracos), dtype=bool)
        )
        mj_raw  = d['melhor_jogada']       if 'melhor_jogada'       in d.files else None
        smj_raw = d['score_melhor_jogada'] if 'score_melhor_jogada' in d.files else None

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    n_padrao     += int((~precisa_adaptativo).sum())
    n_adaptativo += int(precisa_adaptativo.sum())

    if FORCAR_REGRAVAR:
        pendente = np.ones(len(qtd_tracos), dtype=bool)
    else:
        pendente = ~rotulados | (depth_atual < depth_necessaria)

    n_pendentes_padrao     += int((pendente & ~precisa_adaptativo).sum())
    n_pendentes_adaptativo += int((pendente & precisa_adaptativo).sum())

    # Alimenta cache_global com resultados já calculados e suficientes.
    if not FORCAR_REGRAVAR and mj_raw is not None and smj_raw is not None:
        for j in range(len(estados_arr)):
            if rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                chave = (estados_arr[j].tobytes(), int(depth_necessaria[j]))
                if chave not in cache_global:
                    cache_global[chave] = (str(mj_raw[j]), smj_raw[j].copy())

    # Conta tabuleiros distintos pendentes via hash de 64 bits.
    for j in range(len(estados_arr)):
        if pendente[j]:
            eb = estados_arr[j].tobytes()
            h = hash(eb) ^ (int(depth_necessaria[j]) * 0x9E3779B97F4A7C15)
            pending_hashes.add(h)

    if bool(pendente.any()):
        pendentes.append(path)

total_estados   = n_padrao + n_adaptativo
total_pendentes = n_pendentes_padrao + n_pendentes_adaptativo
pct = 100 * total_pendentes / max(total_estados, 1)
n_distintos_pendentes = len(pending_hashes)

print(f'Concluído em {time.time() - t0:.1f}s.')
print(f'\nNPZs originais         : {len(npzs_originais)}')
print(f'Arquivos ja completos  : {len(npzs_originais) - len(pendentes)}')
print(f'Arquivos na fila       : {len(pendentes)}')
print(f'\nTotal de estados       : {total_estados:,}')
print(f'  Depth {DEPTH_PADRAO} (padrao)    : {n_padrao:,}')
print(f'  Depth {DEPTH_ADAPTATIVO} (adaptativo): {n_adaptativo:,}')
print(f'\nPendentes (bruto)      : {total_pendentes:,} ({pct:.2f}%)')
print(f'  com depth {DEPTH_PADRAO}          : {n_pendentes_padrao:,}')
print(f'  com depth {DEPTH_ADAPTATIVO}          : {n_pendentes_adaptativo:,}')
print(f'\nTabuleiros DISTINTOS pendentes : {n_distintos_pendentes:,}')
print(f'Cache global pre-carregado     : {len(cache_global):,} resultados')
print(f'  Tabuleiros identicos nos NPZs novos receberao dados do cache')
print(f'  sem recomputar Minimax.')


Escaneando 417 NPZs (diagnóstico + cache)...
Concluído em 7.4s.

NPZs originais         : 417
Arquivos ja completos  : 411
Arquivos na fila       : 6

Total de estados       : 3,403,460
  Depth 11 (padrao)    : 3,358,837
  Depth 20 (adaptativo): 44,623

Pendentes (bruto)      : 60,000 (1.76%)
  com depth 11          : 59,500
  com depth 20          : 500

Tabuleiros DISTINTOS pendentes : 48,133
Cache global pre-carregado     : 1,957,146 resultados
  Tabuleiros identicos nos NPZs novos receberao dados do cache
  sem recomputar Minimax.


In [3]:
# ── Loop de processamento em lotes ────────────────────────────────────────────
# Para cada lote de NPZs pendentes:
#   1. Coleta estados únicos a calcular (chave = bytes + depth).
#   2. Serve os que já estão no cache_global sem computar.
#   3. Computa via multiprocessing apenas os restantes.
#   4. Adiciona resultados computados ao cache_global (beneficia lotes futuros).
#   5. Grava NPZs atomicamente (.tmp.npz + os.replace).

total_lotes = (len(pendentes) + LOTE_ARQUIVOS - 1) // LOTE_ARQUIVOS

for lote_idx in range(0, len(pendentes), LOTE_ARQUIVOS):
    lote_paths = pendentes[lote_idx : lote_idx + LOTE_ARQUIVOS]
    num_lote = lote_idx // LOTE_ARQUIVOS + 1

    # Coleta estados únicos (chave = bytes + depth) pendentes neste lote.
    estados_para_calcular: dict = {}

    for path in lote_paths:
        with np.load(path, allow_pickle=False) as d:
            qtd_tracos  = d['qtd_tracos'].astype(np.int32)
            qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
            total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
            depth_atual = (
                d['depth_melhor_jogada'].astype(np.int32)
                if 'depth_melhor_jogada' in d.files
                else np.zeros(len(qtd_tracos), dtype=np.int32)
            )
            rotulados = (
                (d['melhor_jogada'] != '')
                if 'melhor_jogada' in d.files
                else np.zeros(len(qtd_tracos), dtype=bool)
            )
            estados_arr = d['estados']

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
        depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

        for j, estado in enumerate(estados_arr):
            if not FORCAR_REGRAVAR and rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                continue
            chave = (estado.tobytes(), int(depth_necessaria[j]))
            estados_para_calcular[chave] = int(depth_necessaria[j])

    print(f'--- Lote {num_lote}/{total_lotes} ({len(lote_paths)} NPZs) ---')
    print(f'    Estados unicos a calcular: {len(estados_para_calcular)}')

    if not estados_para_calcular:
        print('    Nenhum pendente neste lote. Pulando.')
        continue

    # ── Separa: servidos do cache vs precisam de Minimax ──────────────────────
    args_cache: list = []
    args_compute: list = []
    for chave in estados_para_calcular.keys():
        if not FORCAR_REGRAVAR and chave in cache_global:
            args_cache.append(chave)
        else:
            args_compute.append(chave)

    print(f'    Cache global : {len(args_cache):,} | A computar via Minimax: {len(args_compute):,}')

    # Resultados do cache (sem custo computacional).
    resultados_lote: list = [
        (chave[0], chave[1], cache_global[chave][0], cache_global[chave][1].tolist())
        for chave in args_cache
    ]

    # Computa o restante via multiprocessing.
    if args_compute:
        chunksize = max(1, len(args_compute) // (NUM_WORKERS * 16))
        print(f'    Workers: {NUM_WORKERS} | chunksize: {chunksize}')
        t_lote = time.time()
        with multiprocessing.Pool(processes=NUM_WORKERS) as pool:
            it = pool.imap_unordered(processar_estado, args_compute, chunksize=chunksize)
            if HAS_TQDM:
                it = tqdm(it, total=len(args_compute), desc=f'Lote {num_lote}', unit='estado')
            resultados_computados = list(it)

        # Adiciona ao cache_global para beneficiar lotes futuros.
        for eb, dep, mj, sc in resultados_computados:
            chave = (eb, int(dep))
            if chave not in cache_global:
                cache_global[chave] = (mj, np.array(sc, dtype=np.float32))

        resultados_lote.extend(resultados_computados)
        print(f'    {len(resultados_computados):,} computados em {time.time() - t_lote:.1f}s. Gravando...')
    else:
        print(f'    Todos servidos do cache. Gravando...')

    # Cache local do lote: (estado_bytes, depth) → (melhor_jogada, scores_array)
    cache_lote = {
        (eb, int(dep)): (mj, np.array(sc, dtype=np.float32))
        for eb, dep, mj, sc in resultados_lote
    }

    # ── Gravação atômica: preserva TODO o schema v2-a3 ────────────────────────
    for path in lote_paths:
        with np.load(path, allow_pickle=False) as d:
            dados = dict(d)

        estados_arr = dados['estados']
        N = len(estados_arr)
        qtd_tracos  = dados['qtd_tracos'].astype(np.int32)
        qtd_cad     = dados['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = dados['total_caixas_cadeias_longas'].astype(np.int32)

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
        depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

        depth_atual = (
            dados['depth_melhor_jogada'].astype(np.int32)
            if 'depth_melhor_jogada' in dados
            else np.zeros(N, dtype=np.int32)
        )
        rotulados = (
            dados['melhor_jogada'] != ''
            if 'melhor_jogada' in dados
            else np.zeros(N, dtype=bool)
        )

        mj_arr  = np.array(list(dados.get('melhor_jogada', [''] * N)), dtype='<U5')
        smj_arr = dados.get(
            'score_melhor_jogada', np.full((N, 31), -1e9, dtype=np.float32)
        ).copy()
        dm_arr  = depth_atual.copy().astype(np.int8)

        for j, estado in enumerate(estados_arr):
            if not FORCAR_REGRAVAR and rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                continue
            chave = (estado.tobytes(), int(depth_necessaria[j]))
            if chave in cache_lote:
                mj_arr[j], smj_arr[j] = cache_lote[chave]
                dm_arr[j] = np.int8(depth_necessaria[j])

        dados['melhor_jogada']       = mj_arr
        dados['score_melhor_jogada'] = smj_arr
        dados['depth_melhor_jogada'] = dm_arr

        tmp_path = path.with_name(path.stem + '.tmp.npz')
        np.savez_compressed(tmp_path, **dados)
        os.replace(tmp_path, path)
        print(f'    -> {path.name} gravado.')

print('\nTodo o processamento finalizado!')


--- Lote 1/6 (1 NPZs) ---
    Estados unicos a calcular: 8758
    Cache global : 3,348 | A computar via Minimax: 5,410
    Workers: 14 | chunksize: 24


Lote 1:   0%|          | 0/5410 [00:00<?, ?estado/s]

    5,410 computados em 646.3s. Gravando...
    -> dataset_pequeno_0403.npz gravado.
--- Lote 2/6 (1 NPZs) ---
    Estados unicos a calcular: 8773
    Cache global : 3,266 | A computar via Minimax: 5,507
    Workers: 14 | chunksize: 24


Lote 2:   0%|          | 0/5507 [00:00<?, ?estado/s]

    5,507 computados em 636.3s. Gravando...
    -> dataset_pequeno_0404.npz gravado.
--- Lote 3/6 (1 NPZs) ---
    Estados unicos a calcular: 8748
    Cache global : 3,307 | A computar via Minimax: 5,441
    Workers: 14 | chunksize: 24


...

    5,473 computados em 632.3s. Gravando...
    -> dataset_pequeno_0406.npz gravado.
--- Lote 5/6 (1 NPZs) ---
    Estados unicos a calcular: 8753
    Cache global : 3,292 | A computar via Minimax: 5,461
    Workers: 14 | chunksize: 24


Lote 5:   0%|          | 0/5461 [00:00<?, ?estado/s]

    5,461 computados em 625.2s. Gravando...
    -> dataset_pequeno_0407.npz gravado.
--- Lote 6/6 (1 NPZs) ---
    Estados unicos a calcular: 8767
    Cache global : 3,274 | A computar via Minimax: 5,493
    Workers: 14 | chunksize: 24


Lote 6:   0%|          | 0/5493 [00:00<?, ?estado/s]

    5,493 computados em 615.1s. Gravando...
    -> dataset_pequeno_0408.npz gravado.

Todo o processamento finalizado!


In [6]:
# ── Auditoria final ───────────────────────────────────────────────────────────
erros = []
total_padrao = 0
total_adaptativo = 0

for path in npzs_originais:
    with np.load(path, allow_pickle=False) as d:
        if 'melhor_jogada' not in d.files or (d['melhor_jogada'] == '').any():
            erros.append(f'{path.name}: melhor_jogada vazia')
            continue
        dm = d['depth_melhor_jogada'].astype(np.int32)
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    insuficiente = dm < depth_necessaria
    if insuficiente.any():
        erros.append(f'{path.name}: {insuficiente.sum()} estados com depth < necessaria')

    total_padrao     += int((dm == DEPTH_PADRAO).sum())
    total_adaptativo += int((dm == DEPTH_ADAPTATIVO).sum())

if erros:
    print(f'FALHA: {len(erros)} problema(s):')
    for e in erros[:20]:
        print(f'  {e}')
else:
    print(f'OK: {len(npzs_originais)} NPZs auditados.')
    print(f'  Estados com depth {DEPTH_PADRAO} (padrao)    : {total_padrao:,}')
    print(f'  Estados com depth {DEPTH_ADAPTATIVO} (adaptativo): {total_adaptativo:,}')
    print('Pronto para fase4_augmentacao_simetria.ipynb.')


OK: 417 NPZs auditados.
  Estados com depth 11 (padrao)    : 3,358,837
  Estados com depth 20 (adaptativo): 44,623
Pronto para fase4_augmentacao_simetria.ipynb.
